In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/traffic_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

,timestamp,hour,minute,day_of_week,is_weekend,ns_vehicle_count,ew_vehicle_count
0,2025-01-01 00:00:00,0,0,2,0,10,7
1,2025-01-01 00:05:00,0,5,2,0,7,8
2,2025-01-01 00:10:00,0,10,2,0,11,8
3,2025-01-01 00:15:00,0,15,2,0,14,0
4,2025-01-01 00:20:00,0,20,2,0,7,9


In [2]:
# Previous traffic values
for lag in [1, 2, 3, 6, 12]:
    df[f"ns_lag_{lag}"] = df["ns_vehicle_count"].shift(lag)
    df[f"ew_lag_{lag}"] = df["ew_vehicle_count"].shift(lag)

# Rolling averages using only previous values
df["ns_rolling_mean_3"] = df["ns_vehicle_count"].shift(1).rolling(window=3).mean()
df["ew_rolling_mean_3"] = df["ew_vehicle_count"].shift(1).rolling(window=3).mean()

df["ns_rolling_mean_6"] = df["ns_vehicle_count"].shift(1).rolling(window=6).mean()
df["ew_rolling_mean_6"] = df["ew_vehicle_count"].shift(1).rolling(window=6).mean()

df.head(15)

,timestamp,hour,minute,day_of_week,is_weekend,ns_vehicle_count,ew_vehicle_count,ns_lag_1,ew_lag_1,ns_lag_2,...,ns_lag_3,ew_lag_3,ns_lag_6,ew_lag_6,ns_lag_12,ew_lag_12,ns_rolling_mean_3,ew_rolling_mean_3,ns_rolling_mean_6,ew_rolling_mean_6
0,2025-01-01 00:00:00,0,0,2,0,10,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,0,5,2,0,7,8,10.0,7.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:10:00,0,10,2,0,11,8,7.0,8.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 00:15:00,0,15,2,0,14,0,11.0,8.0,7.0,...,10.0,7.0,NaN,NaN,NaN,NaN,9.333333,7.666667,NaN,NaN
4,2025-01-01 00:20:00,0,20,2,0,7,9,14.0,0.0,11.0,...,7.0,8.0,NaN,NaN,NaN,NaN,10.666667,5.333333,NaN,NaN
5,2025-01-01 00:25:00,0,25,2,0,7,7,7.0,9.0,14.0,...,11.0,8.0,NaN,NaN,NaN,NaN,10.666667,5.666667,NaN,NaN
6,2025-01-01 00:30:00,0,30,2,0,14,3,7.0,7.0,7.0,...,14.0,0.0,10.0,7.0,NaN,NaN,9.333333,5.333333,9.333333,6.500000
7,2025-01-01 00:35:00,0,35,2,0,11,3,14.0,3.0,7.0,...,7.0,9.0,7.0,8.0,NaN,NaN,9.333333,6.333333,10.000000,5.833333
8,2025-01-01 00:40:00,0,40,2,0,6,5,11.0,3.0,14.0,...,7.0,7.0,11.0,8.0,NaN,NaN,10.666667,4.333333,10.666667,5.000000
9,2025-01-01 00:45:00,0,45,2,0,10,1,6.0,5.0,11.0,...,14.0,3.0,14.0,0.0,NaN,NaN,10.333333,3.666667,9.833333,4.500000


In [3]:
# Target: traffic in the NEXT 5-minute interval
df["ns_target"] = df["ns_vehicle_count"].shift(-1)
df["ew_target"] = df["ew_vehicle_count"].shift(-1)

# Drop rows with any missing values (from lag features at the start, target at the end)
df = df.dropna().reset_index(drop=True)

df.to_csv("../data/processed/traffic_features.csv", index=False)

print(df.shape)
df.head()

(17267, 23)


,timestamp,hour,minute,day_of_week,is_weekend,ns_vehicle_count,ew_vehicle_count,ns_lag_1,ew_lag_1,ns_lag_2,...,ns_lag_6,ew_lag_6,ns_lag_12,ew_lag_12,ns_rolling_mean_3,ew_rolling_mean_3,ns_rolling_mean_6,ew_rolling_mean_6,ns_target,ew_target
0,2025-01-01 01:00:00,1,0,2,0,9,8,6.0,12.0,6.0,...,14.0,3.0,10.0,7.0,7.333333,8.666667,8.833333,6.166667,0.0,5.0
1,2025-01-01 01:05:00,1,5,2,0,0,5,9.0,8.0,6.0,...,11.0,3.0,7.0,8.0,7.000000,11.000000,8.000000,7.000000,1.0,5.0
2,2025-01-01 01:10:00,1,10,2,0,1,5,0.0,5.0,9.0,...,6.0,5.0,11.0,8.0,5.000000,8.333333,6.166667,7.333333,6.0,9.0
3,2025-01-01 01:15:00,1,15,2,0,6,9,1.0,5.0,0.0,...,10.0,1.0,14.0,0.0,3.333333,6.000000,5.333333,7.333333,4.0,13.0
4,2025-01-01 01:20:00,1,20,2,0,4,13,6.0,9.0,1.0,...,6.0,13.0,7.0,9.0,2.333333,6.333333,4.666667,8.666667,9.0,4.0
